In [1]:
import numpy as np
import itertools
import json
import os
import re
import chipsplitting as cs
from chipsplitting import pairing_matrix, PascalForm, LinearForm
from chipsplitting.hyperfield import HyperfieldVector as HV, HyperfieldHomogeneousLinearSystem as HLinSystem, grid_iter, HyperfieldLinearForm

## Config

In [2]:
CONTRACTION_SIZE = 5
DEGREE = 40
VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

## Load data

In [3]:
DATA = []

def list_files_in_directory(directory):
    files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    return files

def strip_npy_extension(file):
    return file[:-len(".npy")]

directory_path = 'filter' 
files = sorted(list_files_in_directory(directory_path), key=len)

for file in files:
    name = strip_npy_extension(file)
    DATA.append((name, np.load(f"filter/{file}")))
        
print("Data successfully loaded")

Data successfully loaded


## Code 

In [5]:
def absolute(deg, c):
    if type(c) is int:
        return c

    if type(c) is np.str_:
        c = str(c)

    if type(c) is str:
        if len(c) == 3:
            subtrahend = int(c[2])
            if subtrahend >= CONTRACTION_SIZE:
                raise Exception(f"Invalid subtrahend: {c}")
            return deg - int(c[2])
        elif c == 'd':
            return deg
        elif len(c) == 1:
            return int(c)
        else:
            raise Exception(f'string {c} has not length 3')

    raise Exception(f"Invalid type. {c} is of type {type(c)}")

def rel(degree, index):
    assert index < CONTRACTION_SIZE or index > degree - CONTRACTION_SIZE
    return f"d-{degree - index}" if index > CONTRACTION_SIZE else index

def parse_expression(expression):
    if expression[0] != '-':
        expression = '+' + expression
        
    ops = re.findall(r'[\+|-]',expression)
    summands = re.split(r'[\+|-]', expression[1:])
    return (summands, ops)

def realize_expression(degree, mode, op, unit):
    if op == '+':
        return PascalForm(degree, mode, unit)
    elif op == '-':
        return -PascalForm(degree, mode, unit)
    else:
        raise Exception(degree, mode, op, unit)

def form_from_expression(degree, expression, abs_units):
    summands, ops = parse_expression(expression)
    form = LinearForm.zero(degree)
    for mode, op, unit in zip(summands, ops, abs_units):
        form = form + realize_expression(degree, mode, op, unit)
    return form

## -----------------------------------------------------------------
## Prove
## -----------------------------------------------------------------

def prove_zero(d, letter, letter_index, expression, rel_units):
    units = [absolute(d, u) for u in rel_units]
    summands, ops = parse_expression(expression)

    relevant = list(filter(lambda x: filter_relevant(letter, x[0], x[1]), zip(summands, units, ops)))
        
    if not relevant:
        return True
    
    if len(relevant) == 1:
        return True

    # if is homog.
    if [s for s, u, o in relevant] == [relevant[0][0]] * len(relevant):
        expr = ""
        for s, u, op in relevant:
            expr += f"{op}{s}"
        if expr[0] == "+":
            expr = expr[1:]
        units = [u for s, u, op in relevant]
        return prove_homogeneous_form(d, letter, letter_index, expr, units)


    if letter == 'c':
        if letter_index == 1 and (relevant == [("diag",1,"-"), ("row",1,"+")] or relevant == [("diag",1,"+"), ("row",1,"-")]):
            return True

        for mode, unit, op in relevant:
            if letter_index <= unit:
                return False
    elif letter == 'b':
        for mode, unit, op in relevant:
            if mode == "col":
                if letter_index <= unit:
                    return False
            else:
                if letter_index <= d - unit:
                    return False
    elif letter in ('d', 'e', 'd0', 'd1'):
        for mode, unit, op in relevant:
            if letter_index <= d - unit:
                return False
    else:
        raise NotImplementedError()
    
    return True


def prove_constant(d, letter, letter_index, expression, rel_units):
    units = [absolute(d, u) for u in rel_units]
    summands, ops = parse_expression(expression)

    relevant = list(filter(lambda x: filter_relevant(letter, x[0], x[1]), zip(summands, units, ops)))
    
    if not relevant:
        return True

    if len(relevant) == 1:
        return True

    # if is homog.
    if [s for s, u, o in relevant] == [relevant[0][0]] * len(relevant):
        expr = ""
        for s, u, op in relevant:
            expr += f"{op}{s}"
        if expr[0] == "+":
            expr = expr[1:]
        units = [u for s, u, op in relevant]
        return prove_homogeneous_form(d, letter, letter_index, expr, units)



    if letter == 'c':
        if letter_index == 0 and (tuple(relevant) == (("diag",1,"-"), ("row",1,"+")) or tuple(relevant) == (("diag",1,"+"), ("row",1,"-"))):
            return True

        for mode, unit, op in relevant:
            if letter_index < unit:
                return False
    elif letter == 'b':
        for mode, unit, op in relevant:
            if mode == "col":
                if letter_index < unit:
                    return False
            else:
                if letter_index < d - unit:
                    return False
    elif letter in ('d', 'e', 'd0', 'd1'):
        for mode, unit, op in relevant:
            if letter_index < d - unit:
                return False
    else:
        raise NotImplementedError()
    
    return True

def prove_general(d, letter, letter_index, expression, abs_units):
    units = abs_units
    summands, ops = parse_expression(expression)

    def extra_filter(letter, mode, unit):
        if letter in ("b", "c"):
            return unit > letter_index
        else:
            return d - unit > letter_index

    relevant = list(filter(lambda x: filter_relevant(letter, x[0], x[1]) and extra_filter(letter, x[0], x[1]), zip(summands, units, ops)))
    
    if not relevant:
        assert False, f"Cannot happen. {relevant}"

    if len(relevant) == 1:
        return True

    # if is homog.
    if [s for s, u, o in relevant] == [relevant[0][0]] * len(relevant):
        expr = ""
        for s, u, op in relevant:
            expr += f"{op}{s}"
        if expr[0] == "+":
            expr = expr[1:]
        units = [u for s, u, op in relevant]
        return prove_homogeneous_form(d, letter, letter_index, expr, units)

    has_pos_sign = False
    has_neg_sign = False
    
    if letter == 'c':
        for m,u,op in relevant:
            if m == "diag":
                if op == "+":
                    has_pos_sign = True
                else:
                    has_neg_sign = True
            elif m == "row":
                if op == "+":
                    if (u - letter_index) % 2 == 0:
                        has_pos_sign = True
                    else:
                        has_neg_sign = True
                else:
                    if (u - letter_index) % 2 == 0:
                        has_neg_sign = True
                    else:
                        has_pos_sign = True
                        
            if has_pos_sign and has_neg_sign:
                return False
    elif letter == 'b':
        for m,u,op in relevant:
            if m == "diag":
                if op == "+":
                    has_pos_sign = True
                else:
                    has_neg_sign = True
            elif m == "col":
                if op == "+":
                    if (u - letter_index) % 2 == 0:
                        has_pos_sign = True
                    else:
                        has_neg_sign = True
                else:
                    if (u - letter_index) % 2 == 0:
                        has_neg_sign = True
                    else:
                        has_pos_sign = True
                        
            if has_pos_sign and has_neg_sign:
                return False
    elif letter == 'd':
        for m,u,op in relevant:
            if m == "row":
                sign = absolute(d, u) + CONTRACTION_SIZE
            else:
                sign = absolute(d, u) + (d - CONTRACTION_SIZE - letter_index)
            if sign % 2 == 0:
                if op == "+":
                    has_pos_sign = True
                else:
                    has_neg_sign = True
            else:
                if op == "+":
                    has_neg_sign = True
                else:
                    has_pos_sign = True
            if has_pos_sign and has_neg_sign:
                return False
    elif letter == 'e':
        for m,u,op in relevant:
            if m == "row":
                sign = absolute(d, u) + CONTRACTION_SIZE + 1
            else:
                sign = absolute(d, u) + (d - CONTRACTION_SIZE - 1 - letter_index)
            if sign % 2 == 0:
                if op == "+":
                    has_pos_sign = True
                else:
                    has_neg_sign = True
            else:
                if op == "+":
                    has_neg_sign = True
                else:
                    has_pos_sign = True
            if has_pos_sign and has_neg_sign:
                return False        

    return True

def is_expression_homogeneous(expression):
    summands, ops = parse_expression(expression)
    return summands == [summands[0]] * len(summands)

def filter_relevant(letter, summand, unit):
    if letter == 'c':
        return summand in ('row', 'diag') and unit < CONTRACTION_SIZE
    elif letter == 'b':
        return (summand == 'col' and unit < CONTRACTION_SIZE) or (summand == 'diag' and unit > CONTRACTION_SIZE)
    elif letter in ('d', 'e', 'd0', 'd1'):
        return summand in ('row', 'col') and unit > CONTRACTION_SIZE
    else:
        raise NotImplementedError(f"{summand} {unit}")

def prove_homogeneous_form(degree, letter, letter_index, expression, abs_units):
    summands, ops = parse_expression(expression)
    form = form_from_expression(degree, expression, abs_units)

    relevant = list(filter(
        lambda x: filter_relevant(letter, x[0], x[2]), 
        zip(summands, ops, abs_units)
    ))

    counts = {u: {'+': 0, '-': 0} for s, o, u in relevant}

    for s, o, u in relevant:
        counts[u][o] += 1

    new_relevant = []
    for unit, count in counts.items():
        for _ in range(abs(count["+"] - count["-"])):
            new_relevant.append((summands[0], "+" if count["+"] - count["-"] > 0 else "-", unit))
    relevant = new_relevant

    if not relevant:
        return True

    dominant_summand, dominant_op, dominant_unit = max(relevant, key=lambda x: x[2] if x[2] < CONTRACTION_SIZE else degree - x[2] )
    all_dominants = [(s, op, u) for s, op, u in relevant if u == dominant_unit]


    if len([op for s, op, u in all_dominants if op != dominant_op]) > 0:
        print(f"Cant prove it here: expr={expression}, units={abs_units}, dominant_unit={dominant_unit}")
        return False
    
    dominant_form = realize_expression(degree, dominant_summand, dominant_op, dominant_unit)
    
    res = np.all(
        np.sign(dominant_form.get(CONTRACTION_SIZE, f"{letter}_{letter_index}")) == np.sign(form.get(CONTRACTION_SIZE, f"{letter}_{letter_index}"))
    )

    return res

def mirror_mode(m):
    if m == "row":
        return "col"
    if m == "col":
        return "row"
    return "diag"

def mirror_unit(u):
    if len(u) == 1:
        return f"d-{u}"
    return u[2]

def mirror(expr, unit_list):
    summands, ops = parse_expression(expr)
    summands = [mirror_mode(m) for m in summands]
    indexes_of_diag = [i for i, x in enumerate(summands) if x == "diag"]
    unit_list = [[str(u) if i not in indexes_of_diag else mirror_unit(u) for i, u in enumerate(units)] for units in unit_list]

    expr = summands[0] if ops[0] == "+" else f"-{summands[0]}"

    for summand, op in zip(summands[1:], ops[1:]):
        expr += f"{op}{summand}"
    
    return (expr, unit_list)
    


## Proof c-contractable ✅

We prove that all the Pascal templates defined in DATA are fixed-contractable on c. It suffices to show it for either even or odd degrees since the even-degree and odd-degree contractions coincide. 

In [15]:
%%time

VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

failed = []
d = DEGREE

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)
        hyper_p = p.to_hyperfield()
        
        # already verified, so we skip it
        if hyper_p in VERIFIED: 
            continue

        if p == LinearForm.zero(p.degree):
            continue

        if p.support_pos[0] or p.support_neg[0]:
            continue
    
        for i in range(CONTRACTION_SIZE):
            # check c
            c = p.get(CONTRACTION_SIZE, f"c_{i}")
            
            if is_expression_homogeneous(expression):
                if prove_homogeneous_form(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("homo: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == 0):
                if prove_zero(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("zero: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == c[0]):
                if prove_constant(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("constant: ", (expression, units, i))
                    failed.append((expression, units, i))
            else:
                if prove_general(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("general: ", (expression, units, i))
                    failed.append((expression, units, i))
            
                
print(f"Completed. {len(failed)} cases failed.")
failed


row+row [2, 3]
row + 3
[  -10   -20   -35   -56   -84  -120  -165  -220  -286  -364  -455  -560
  -680  -816  -969 -1140 -1330 -1540 -1771 -2024 -2300 -2600 -2925 -3276
 -3654 -4060 -4495 -4960 -5456 -5984 -6545]
c 0
[    0    -5   -14   -28   -48   -75  -110  -154  -208  -273  -350  -440
  -544  -663  -798  -950 -1120 -1309 -1518 -1748 -2000 -2275 -2574 -2898
 -3248 -3625 -4030 -4464 -4928 -5423 -5950]

general:  ('row+row+col-diag', (2, 3, 0, 0), 0)
constant:  ('diag-diag+diag+row-col', (0, 1, 39, 1, 0), 0)
zero:  ('diag-diag+diag+row-col', (0, 1, 39, 1, 0), 1)
Completed. 3 cases failed.
CPU times: user 38min 9s, sys: 2.23 s, total: 38min 12s
Wall time: 38min 13s


[('row+row+col-diag', (2, 3, 0, 0), 0),
 ('diag-diag+diag+row-col', (0, 1, 39, 1, 0), 0),
 ('diag-diag+diag+row-col', (0, 1, 39, 1, 0), 1)]

In [25]:
A = failed

A = [tuple(zip(*parse_expression(expr), units)) for expr, units, _ in A]
A = [[(mode, op, unit) for mode, op, unit in expr if filter_relevant('c', mode, unit)] for expr in A]
A

[[('row', '+', 2), ('row', '+', 3), ('diag', '-', 0)],
 [('diag', '+', 0), ('diag', '-', 1), ('row', '+', 1)],
 [('diag', '+', 0), ('diag', '-', 1), ('row', '+', 1)]]

## Proof b-contractable ✅

In [6]:
%%time

VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
    + [-PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

failed = []
d = DEGREE


for expression, units_list in [mirror(expr, unit_list) for expr, unit_list in DATA]:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)
        hyper_p = p.to_hyperfield()
        
        # already verified, so we skip it
        if hyper_p in VERIFIED or -hyper_p in VERIFIED: 
            continue

        if p == LinearForm.zero(p.degree):
            continue

        if p.support_pos[0] or p.support_neg[0]:
            continue
    
        for i in range(CONTRACTION_SIZE):
            # check c
            c = p.get(CONTRACTION_SIZE, f"c_{i}")
            
            if is_expression_homogeneous(expression):
                if prove_homogeneous_form(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("homo: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == 0):
                if prove_zero(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("zero: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == c[0]):
                if prove_constant(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("constant: ", (expression, units, i))
                    failed.append((expression, units, i))
            else:
                if prove_general(d, "c", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("gen: ", (expression, units, i))
                    failed.append((expression, units, i))
            
                
print(f"Completed. {len(failed)} cases failed.")
failed

gen:  ('diag-diag+row+col+row', (40, 0, 2, 1, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 2, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 3, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 4, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 36, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 37, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 38, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 39, 3), 0)
gen:  ('diag-diag+row+col+row', (40, 0, 2, 40, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 2, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 4, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 36, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 37, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 38, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 39, 3), 0)
gen:  ('diag-diag+row+col+row', (39, 1, 2, 40, 3), 0)
gen:  ('diag-diag+row+col+row', (1, 39, 1, 1, 2), 0)
gen:  ('diag-diag+row+col+row', (1, 39, 1, 1, 4), 0)
gen:  ('diag-diag+row+col+row', (1, 

[('diag-diag+row+col+row', (40, 0, 2, 1, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 2, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 3, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 4, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 36, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 37, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 38, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 39, 3), 0),
 ('diag-diag+row+col+row', (40, 0, 2, 40, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 2, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 4, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 36, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 37, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 38, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 39, 3), 0),
 ('diag-diag+row+col+row', (39, 1, 2, 40, 3), 0),
 ('diag-diag+row+col+row', (1, 39, 1, 1, 2), 0),
 ('diag-diag+row+col+row', (1, 39, 1, 1, 4), 0),
 ('diag-diag+row+col+row', (1, 39, 1, 3, 2), 0),
 ('diag-diag+row+col+row', (1, 39, 1, 3, 4), 0),
 ('diag-di

In [16]:
D = 16
p = PascalForm(D, "diag", 1) + PascalForm(D, "row", 1) + PascalForm(D, "row", 2)
print(p)


p = PascalForm(D, "row", 1) + PascalForm(D, "row", 2)
print(p)

 104 
  91  -13 
  79  -12    1 
  68  -11    1    . 
  58  -10    1    .    . 
  49   -9    1    .    .    . 
  41   -8    1    .    .    .    . 
  34   -7    1    .    .    .    .    . 
  28   -6    1    .    .    .    .    .    . 
  23   -5    1    .    .    .    .    .    .    . 
  19   -4    1    .    .    .    .    .    .    .    . 
  16   -3    1    .    .    .    .    .    .    .    .    . 
  14   -2    1    .    .    .    .    .    .    .    .    .    . 
  13   -1    1    .    .    .    .    .    .    .    .    .    .    . 
  13    .    1    .    .    .    .    .    .    .    .    .    .    .    . 
  14    1    1    .    .    .    .    .    .    .    .    .    .    .    .    . 
  16    2    1    .    .    .    .    .    .    .    .    .    .    .    .    .    . 

 104 
  90  -14 
  77  -13    1 
  65  -12    1    . 
  54  -11    1    .    . 
  44  -10    1    .    .    . 
  35   -9    1    .    .    .    . 
  27   -8    1    .    .    .    .    . 
  20   -7    1    .    .    .

In [8]:
A = [tuple(zip(*parse_expression(expr), units)) for expr, units, _ in failed]
A = list(set([tuple([(mode, op, unit) for mode, op, unit in expr if filter_relevant('c', mode, unit)]) for expr in A]))
A

[(('diag', '-', 0), ('row', '+', 2), ('row', '+', 3)),
 (('diag', '+', 1), ('row', '+', 4), ('row', '+', 4), ('row', '-', 2)),
 (('diag', '-', 1), ('row', '+', 2), ('row', '+', 3)),
 (('diag', '-', 0), ('row', '+', 1), ('row', '+', 2), ('row', '-', 4)),
 (('diag', '+', 0), ('row', '+', 3), ('row', '+', 4), ('row', '-', 1)),
 (('diag', '-', 1), ('row', '+', 1), ('row', '+', 2), ('row', '-', 4)),
 (('diag', '+', 1), ('row', '+', 1), ('row', '+', 1), ('row', '-', 3)),
 (('diag', '-', 1), ('row', '+', 2), ('row', '+', 3), ('row', '-', 1)),
 (('diag', '+', 1), ('row', '+', 1), ('row', '+', 4)),
 (('diag', '-', 1), ('row', '+', 1), ('row', '-', 0)),
 (('diag', '+', 0), ('row', '+', 4), ('row', '+', 4), ('row', '-', 2)),
 (('diag', '+', 1), ('row', '+', 4), ('row', '-', 2)),
 (('diag', '+', 1), ('row', '+', 1), ('row', '+', 2), ('row', '-', 3)),
 (('diag', '+', 1), ('row', '+', 1), ('row', '-', 3)),
 (('diag', '-', 1), ('row', '+', 3), ('row', '+', 3), ('row', '-', 1)),
 (('diag', '+', 1), ('

In [9]:
# apply prop:row_homo_zero_diag
only_diag_0 = [f for f in A if ('diag', '+', 0) in f or ('diag', '-', 0) in f]

for f in only_diag_0:
    expr = ""
    op_diag = ""
    for s, op, u in f:
        if s != "diag":
            expr += f"{op}{s}"
        else:
            op_diag = op
    if expr[0] == "+":
        expr = expr[1:]
    units = [u for s, op, u in f if s != "diag"]
    p = form_from_expression(d, expr, units)
    c = p.get(CONTRACTION_SIZE, "c_0")
    if op_diag == "+":
        if not np.all(c>=0):
            print("Fail")
    elif op_diag == "-":
        if not np.all(c <= 0):
            print("Fail")
    else:
        assert False, "This cant happen"

In [18]:
only_diag_1 = [f for f in A if ('diag', '+', 1) in f or ('diag', '-', 1) in f]

for f in only_diag_1:
    expr = ""
    op_diag = ""
    unit_diag = ""
    for s, op, u in f:
        if s != "diag":
            expr += f"{op}{s}"
        else:
            op_diag = op
            unit_diag = u
    if expr[0] == "+":
        expr = expr[1:]

    units = [u for s, op, u in f if s != "diag"]
    p = form_from_expression(d, expr, units)
    dominant_summand, dominant_op, dominant_unit = max([x for x in f if x[0] != "diag"], key=lambda x: x[2])

    
    c0 = p.get(CONTRACTION_SIZE, "c_0")
    c1 = p.get(CONTRACTION_SIZE, "c_1")

    diag = PascalForm(d, "diag", unit_diag)
    if op_diag == "-":
        diag = -diag

    # props:fixeed-contractable-unirow-ui23
    if ((np.all(diag.get(CONTRACTION_SIZE, "c_0") > 0) and np.all(c0 > 0)) or (np.all(diag.get(CONTRACTION_SIZE, "c_0") < 0) and np.all(c0 < 0))) and (np.all(c1 > 1) or np.all(c1 < -1)):
        pass
    else:
        print(f)
        dominant_form = PascalForm(d, dominant_summand, dominant_unit)
        if dominant_op == '-':
            dominant_form = -dominant_form
        dominant_c0 = dominant_form.get(CONTRACTION_SIZE, "c_0")
        if not np.all(np.sign(dominant_c0) == np.sign(diag.get(CONTRACTION_SIZE, "c_0"))) or not np.all(np.abs(c1) >= 1):
            print("fail")


(('diag', '+', 1), ('row', '+', 4), ('row', '+', 4), ('row', '-', 2))
(('diag', '-', 1), ('row', '+', 2), ('row', '+', 3))
(('diag', '-', 1), ('row', '+', 1), ('row', '+', 2), ('row', '-', 4))
(('diag', '+', 1), ('row', '+', 1), ('row', '+', 1), ('row', '-', 3))
(('diag', '-', 1), ('row', '+', 2), ('row', '+', 3), ('row', '-', 1))
(('diag', '+', 1), ('row', '+', 1), ('row', '+', 4))
(('diag', '-', 1), ('row', '+', 1), ('row', '-', 0))
(('diag', '+', 1), ('row', '+', 4), ('row', '-', 2))
(('diag', '+', 1), ('row', '+', 1), ('row', '+', 4), ('row', '-', 2))
(('diag', '-', 1), ('row', '+', 2), ('row', '-', 4))


## Proof d-contractable ✅

### even degree ✅

In [6]:
%%time

VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

failed = []
d = DEGREE

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)
        hyper_p = p.to_hyperfield()
        
        # already verified, so we skip it
        if hyper_p in VERIFIED: 
            continue

        if p == LinearForm.zero(p.degree):
            continue

        if p.support_pos[0] or p.support_neg[0]:
            continue
    
        for i in range(CONTRACTION_SIZE):
            # check d
            c = p.get(CONTRACTION_SIZE, f"d_{i}")
            
            if is_expression_homogeneous(expression):
                if prove_homogeneous_form(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("homo: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == 0):
                if prove_zero(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("zero: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == c[0]):
                if prove_constant(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("constant: ", (expression, units, i))
                    failed.append((expression, units, i))
            else:
                if prove_general(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("gen: ", (expression, units, i))
                    failed.append((expression, units, i))
            
                
print(f"Completed. {len(failed)} cases failed.")
failed

zero:  ('col-row-col', (40, 40, 1), 0)
zero:  ('col-row-col', (40, 40, 2), 0)
zero:  ('col-row-col', (40, 40, 3), 0)
zero:  ('col-row-col', (40, 40, 4), 0)
zero:  ('diag-diag+col+row-col', (1, 39, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (1, 39, 3, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (2, 38, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (3, 37, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (4, 36, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (39, 1, 2, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (39, 1, 4, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 2, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 3, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 4, 40, 40), 0)

col+col [37, 38]
col + 37
[  10   35   84  165  286  455  680  969 1330 1771 2300 2925 3654 4495
 5456 6545]
d 0
[   0   14   48  110  208  350  544  798 1120 1518 2000 2574 3248 4030
 4928 5950]

gen:  ('diag-diag+col+col-col'

[('col-row-col', (40, 40, 1), 0),
 ('col-row-col', (40, 40, 2), 0),
 ('col-row-col', (40, 40, 3), 0),
 ('col-row-col', (40, 40, 4), 0),
 ('diag-diag+col+row-col', (1, 39, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (1, 39, 3, 40, 40), 0),
 ('diag-diag+col+row-col', (2, 38, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (3, 37, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (4, 36, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (39, 1, 2, 40, 40), 0),
 ('diag-diag+col+row-col', (39, 1, 4, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 2, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 3, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 4, 40, 40), 0),
 ('diag-diag+col+col-col', (0, 40, 37, 38, 40), 0),
 ('diag-diag+col+col-col', (1, 39, 37, 38, 40), 0),
 ('diag-diag+col+col-col', (2, 38, 37, 38, 40), 0),
 ('diag-diag+col+col-col', (3, 37, 37, 38, 40), 0),
 ('diag-diag+col+col-col', (4, 36, 37, 38, 40), 0),
 ('diag-diag+col+col-col', (36, 4, 37, 38, 

In [20]:
A = [tuple(zip(*parse_expression(expr), units)) for expr, units, _ in failed]
A = list(set([tuple([(mode, op, unit) for mode, op, unit in expr if filter_relevant('d', mode, unit)]) for expr in A]))
A

[(('col', '+', 40), ('row', '-', 40)),
 (('col', '+', 37), ('col', '+', 38), ('col', '-', 40)),
 (('row', '+', 40), ('col', '-', 40))]

The first and third forms are zero forms. The second form is verified in the master's thesis.

### odd degree ✅

In [9]:
%%time

DEGREE = 41

VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

failed = []
d = DEGREE

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)
        hyper_p = p.to_hyperfield()
        
        # already verified, so we skip it
        if hyper_p in VERIFIED: 
            continue

        if p == LinearForm.zero(p.degree):
            continue

        if p.support_pos[0] or p.support_neg[0]:
            continue
    
        for i in range(CONTRACTION_SIZE):
            # check d
            c = p.get(CONTRACTION_SIZE, f"d_{i}")
            
            if is_expression_homogeneous(expression):
                if prove_homogeneous_form(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("homo: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == 0):
                if prove_zero(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("zero: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == c[0]):
                if prove_constant(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("constant: ", (expression, units, i))
                    failed.append((expression, units, i))
            else:
                if prove_general(d, "d", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("gen: ", (expression, units, i))
                    failed.append((expression, units, i))
            
DEGREE = 40

print(f"Completed. {len(failed)} cases failed.")
failed

zero:  ('col+row-col', (41, 41, 1), 0)
zero:  ('col+row-col', (41, 41, 2), 0)
zero:  ('col+row-col', (41, 41, 3), 0)
zero:  ('col+row-col', (41, 41, 4), 0)
zero:  ('diag-diag+col+row+col', (1, 40, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (1, 40, 3, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (2, 39, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (3, 38, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (4, 37, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (40, 1, 2, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (40, 1, 4, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 2, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 3, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 4, 41, 41), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 1), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 2), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 3), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 4), 0)
zero

[('col+row-col', (41, 41, 1), 0),
 ('col+row-col', (41, 41, 2), 0),
 ('col+row-col', (41, 41, 3), 0),
 ('col+row-col', (41, 41, 4), 0),
 ('diag-diag+col+row+col', (1, 40, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (1, 40, 3, 41, 41), 0),
 ('diag-diag+col+row+col', (2, 39, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (3, 38, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (4, 37, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (40, 1, 2, 41, 41), 0),
 ('diag-diag+col+row+col', (40, 1, 4, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 2, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 3, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 4, 41, 41), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 1), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 2), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 3), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 4), 0),
 ('diag-diag+col+row-col', (1, 40, 41, 41, 2), 0),
 ('diag-diag+col+row-col', (1, 40, 41, 41, 4), 0

In [10]:
A = [tuple(zip(*parse_expression(expr), units)) for expr, units, _ in failed]
A = list(set([tuple([(mode, op, unit) for mode, op, unit in expr if filter_relevant('d', mode, unit)]) for expr in A]))
A

[(('col', '+', 41), ('row', '+', 41)),
 (('row', '+', 41), ('col', '+', 41)),
 (('col', '+', 38), ('col', '+', 39), ('col', '-', 41))]

The first and third forms are zero forms. The second form is verified in the master's thesis.

## Proof e-contractable ✅

### even degree ✅

In [33]:
%%time

DEGREE = 40
VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

failed = []
d = DEGREE

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)
        hyper_p = p.to_hyperfield()
        
        # already verified, so we skip it
        if hyper_p in VERIFIED: 
            continue

        if p == LinearForm.zero(p.degree):
            continue

        if p.support_pos[0] or p.support_neg[0]:
            continue
    
        for i in range(CONTRACTION_SIZE):
            # check e
            c = p.get(CONTRACTION_SIZE, f"e_{i}")
            
            if is_expression_homogeneous(expression):
                if prove_homogeneous_form(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("homo: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == 0):
                if prove_zero(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("zero: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == c[0]):
                if prove_constant(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("constant: ", (expression, units, i))
                    failed.append((expression, units, i))
            else:
                if prove_general(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("gen: ", (expression, units, i))
                    failed.append((expression, units, i))
            
                
print(f"Completed. {len(failed)} cases failed.")
failed

zero:  ('col-row-col', (40, 40, 1), 0)
zero:  ('col-row-col', (40, 40, 2), 0)
zero:  ('col-row-col', (40, 40, 3), 0)
zero:  ('col-row-col', (40, 40, 4), 0)
zero:  ('diag-diag+col+row-col', (1, 39, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (1, 39, 3, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (2, 38, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (3, 37, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (4, 36, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (39, 1, 2, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (39, 1, 4, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 1, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 2, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 3, 40, 40), 0)
zero:  ('diag-diag+col+row-col', (40, 0, 4, 40, 40), 0)
zero:  ('diag-diag+col-row-col', (0, 40, 40, 40, 1), 0)
zero:  ('diag-diag+col-row-col', (0, 40, 40, 40, 4), 0)
zero:  ('diag-diag+col-row-col', (1, 39, 40, 40, 4), 0)
zero:  ('diag-diag+col-row-col', (36, 4, 40, 40, 1), 0)
zero

[('col-row-col', (40, 40, 1), 0),
 ('col-row-col', (40, 40, 2), 0),
 ('col-row-col', (40, 40, 3), 0),
 ('col-row-col', (40, 40, 4), 0),
 ('diag-diag+col+row-col', (1, 39, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (1, 39, 3, 40, 40), 0),
 ('diag-diag+col+row-col', (2, 38, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (3, 37, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (4, 36, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (39, 1, 2, 40, 40), 0),
 ('diag-diag+col+row-col', (39, 1, 4, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 1, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 2, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 3, 40, 40), 0),
 ('diag-diag+col+row-col', (40, 0, 4, 40, 40), 0),
 ('diag-diag+col-row-col', (0, 40, 40, 40, 1), 0),
 ('diag-diag+col-row-col', (0, 40, 40, 40, 4), 0),
 ('diag-diag+col-row-col', (1, 39, 40, 40, 4), 0),
 ('diag-diag+col-row-col', (36, 4, 40, 40, 1), 0),
 ('diag-diag+col-row-col', (37, 3, 40, 40, 1), 0),
 ('diag-diag+col-row-col', (38, 2, 40, 40, 1), 0

In [34]:
A = [tuple(zip(*parse_expression(expr), units)) for expr, units, _ in failed]
A = list(set([tuple([(mode, op, unit) for mode, op, unit in expr if filter_relevant('d', mode, unit)]) for expr in A]))
A

[(('row', '+', 40), ('col', '-', 40)), (('col', '+', 40), ('row', '-', 40))]

This Pascal form is zero.

In [36]:
(PascalForm(d, "row", d) - PascalForm(d, "col", d)) == LinearForm.zero(d)

np.True_

### odd degree ✅¶

In [37]:
%%time

DEGREE = 41
VERIFIED = set(
    [LinearForm.zero(DEGREE).to_hyperfield()] 
    + [PascalForm(DEGREE, m, k).to_hyperfield() for k in list(range(CONTRACTION_SIZE)) + list(range(DEGREE-CONTRACTION_SIZE+1, DEGREE+1)) for m in ['row', 'col', 'diag']]
)

failed = []
d = DEGREE

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)
        hyper_p = p.to_hyperfield()
        
        # already verified, so we skip it
        if hyper_p in VERIFIED: 
            continue

        if p == LinearForm.zero(p.degree):
            continue

        if p.support_pos[0] or p.support_neg[0]:
            continue
    
        for i in range(CONTRACTION_SIZE):
            # check e
            c = p.get(CONTRACTION_SIZE, f"e_{i}")
            
            if is_expression_homogeneous(expression):
                if prove_homogeneous_form(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("homo: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == 0):
                if prove_zero(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("zero: ", (expression, units, i))
                    failed.append((expression, units, i))
            elif np.all(c == c[0]):
                if prove_constant(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("constant: ", (expression, units, i))
                    failed.append((expression, units, i))
            else:
                if prove_general(d, "e", i, expression, units):
                    VERIFIED.add(hyper_p)
                else:
                    print("gen: ", (expression, units, i))
                    failed.append((expression, units, i))
            
DEGREE = 40
                
print(f"Completed. {len(failed)} cases failed.")
failed

zero:  ('col+row-col', (41, 41, 1), 0)
zero:  ('col+row-col', (41, 41, 2), 0)
zero:  ('col+row-col', (41, 41, 3), 0)
zero:  ('col+row-col', (41, 41, 4), 0)
zero:  ('diag-diag+col+row+col', (1, 40, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (1, 40, 3, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (2, 39, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (3, 38, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (4, 37, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (40, 1, 2, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (40, 1, 4, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 1, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 2, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 3, 41, 41), 0)
zero:  ('diag-diag+col+row+col', (41, 0, 4, 41, 41), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 1), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 2), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 3), 0)
zero:  ('diag-diag+col+row-col', (0, 41, 41, 41, 4), 0)
zero

[('col+row-col', (41, 41, 1), 0),
 ('col+row-col', (41, 41, 2), 0),
 ('col+row-col', (41, 41, 3), 0),
 ('col+row-col', (41, 41, 4), 0),
 ('diag-diag+col+row+col', (1, 40, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (1, 40, 3, 41, 41), 0),
 ('diag-diag+col+row+col', (2, 39, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (3, 38, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (4, 37, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (40, 1, 2, 41, 41), 0),
 ('diag-diag+col+row+col', (40, 1, 4, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 1, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 2, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 3, 41, 41), 0),
 ('diag-diag+col+row+col', (41, 0, 4, 41, 41), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 1), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 2), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 3), 0),
 ('diag-diag+col+row-col', (0, 41, 41, 41, 4), 0),
 ('diag-diag+col+row-col', (1, 40, 41, 41, 2), 0),
 ('diag-diag+col+row-col', (1, 40, 41, 41, 4), 0

In [38]:
A = [tuple(zip(*parse_expression(expr), units)) for expr, units, _ in failed]
A = list(set([tuple([(mode, op, unit) for mode, op, unit in expr if filter_relevant('d', mode, unit)]) for expr in A]))
A

[(('col', '+', 41), ('row', '+', 41)), (('row', '+', 41), ('col', '+', 41))]

This is Pascal form is zero.

In [39]:
(PascalForm(41, "col", 41) + PascalForm(41, "row", 41)) == LinearForm.zero(41)

np.True_